In [ ]:
!pip install -U transformers

Preparing dataset

use the CoNLL-2003 dataset

In [ ]:
from datasets import load_dataset
dataset = load_dataset("lhoestq/conll2003")

In [ ]:
dataset

In [ ]:
dataset["train"][0]["tokens"]

using ner labels

In [ ]:
ner_feature = dataset["train"].features["ner_tags"]
label_name = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-MISC", "I-MISC"
]


In [ ]:
dataset["train"][0]["ner_tags"]
# ner_feature = dataset["train"].features["ner_tags"]

In [ ]:
words = dataset["train"][0]["tokens"]
labels = dataset["train"][0]["ner_tags"]
line1 = ""
line2 = ""
for word, label in zip(words, labels):
  full_label = label_name[label]
  max_length = max(len(word), len(full_label))
  line1 += word + " " * (max_length - len(word) + 1)
  line2 += full_label + " " * (max_length - len(full_label) + 1)

print(line1)
print(line2)

using pos and chunk labels

In [ ]:
pos_features = dataset["train"].features['pos_tags']
# pos_features
pos_labels = [
    '"', "''", "#", "$", "(", ")", ",", ".", ":", "``",
    "CC", "CD", "DT", "EX", "FW", "IN", "JJ", "JJR", "JJS",
    "LS", "MD", "NN", "NNP", "NNPS", "NNS", "PDT", "POS",
    "PRP", "PRP$", "RB", "RBR", "RBS", "RP", "SYM", "TO",
    "UH", "VB", "VBD", "VBG", "VBN", "VBP", "VBZ", "WDT",
    "WP", "WP$", "WRB"
]
chunk_labels =[
    "O",
    "B-ADJP", "I-ADJP",
    "B-ADVP", "I-ADVP",
    "B-CONJP", "I-CONJP",
    "B-INTJ", "I-INTJ",
    "B-LST", "I-LST",
    "B-NP", "I-NP",
    "B-PP", "I-PP",
    "B-PRT", "I-PRT",
    "B-SBAR", "I-SBAR",
    "B-UCP", "I-UCP",
    "B-VP", "I-VP"
]

In [ ]:
ins_pos_label = dataset["train"][0]["pos_tags"]
ins_chunk_label = dataset["train"][0]["chunk_tags"]
line1_pos = ""
line1_chunk = ""
line2_pos = ""
line2_chunk = ""
for word, label in zip(words, ins_pos_label):
  full_label = pos_labels[label]
  max_length = max(len(word), len(full_label))
  line1_pos += word + " " * (max_length - len(word) + 1)
  line2_pos += full_label + " " * (max_length - len(full_label) + 1)

print(line1_pos)
print(line2_pos)

for word, label in zip(words, ins_chunk_label):
  full_label1 = chunk_labels[label]
  max_length = max(len(word), len(full_label1))
  line1_chunk += word + " " * (max_length - len(word) + 1)
  line2_chunk += full_label1 + " " * (max_length - len(full_label1) + 1)
print("token of chunk")
print(line1_chunk)
print(line2_chunk)

# Processing dataset

In [ ]:
from transformers import AutoTokenizer
model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
inputs = tokenizer(dataset["train"][0]["tokens"], is_split_into_words= True)
inputs.tokens()

ome researchers prefer to attribute only one label per word, and assign -100 to the other subtokens in a given word. This is to avoid long words that split into lots of subtokens contributing heavily to the loss. Change the previous function to align labels with input IDs by following this rule.

In [ ]:
def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id != current_word:
            # Start of a new word!
            current_word = word_id
            label = -100 if word_id is None else labels[word_id]
            new_labels.append(label)
        elif word_id is None:
            # Special token
            new_labels.append(-100)
        else:
            # Same word as previous token
            label = labels[word_id]
            # If the label is B-XXX we change it to I-XXX
            if label % 2 == 1:
                label += 1
            new_labels.append(label)

    return new_labels

In [ ]:
labels = dataset["train"][100]["ner_tags"]
word_ids = inputs.word_ids()
print(labels)
print(align_labels_with_tokens(labels, word_ids))

In [ ]:
def tokenize_align_labels(example):
  tokenized_inputs = tokenizer(example["tokens"], truncation = True, is_split_into_words= True)
  all_labels = example["ner_tags"]
  new_labels = []
  for i, labels in enumerate(all_labels):
    word_ids = tokenized_inputs.word_ids(i)
    new_labels.append(align_labels_with_tokens(labels, word_ids))

  tokenized_inputs["labels"] = new_labels
  return tokenized_inputs

In [ ]:
tokenized_datasets = dataset.map(
    tokenize_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

# Fine-tuning the model with the Trainer API

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
batch = data_collator([tokenized_datasets["train"][i] for i in range(2)])
batch["labels"]

In [ ]:
!pip install seqeval
!pip install evaluate


In [ ]:
import evaluate

metric = evaluate.load("seqeval")

In [ ]:
labels = dataset["train"][0]["ner_tags"]
labels = [label_name[i] for i in labels]
labels

In [ ]:
predictions = labels.copy()
predictions[2] = "O"
metric.compute(predictions=[predictions], references=[labels])

In [ ]:
import numpy as np


def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_name[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_name[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

In [ ]:
id2label = {i: label for i, label in enumerate(label_name)}
label2id = {v: k for k, v in id2label.items()}

load model

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id,
)

In [ ]:
model.config.num_labels

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="bert-finetuned-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)
trainer.train()

In [ ]:
trainer.push_to_hub(commit_message="bert-finetuned-ner")

# custom training loop

#prepare anything

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True,
    collate_fn=data_collator,
    batch_size=8,
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], collate_fn=data_collator, batch_size=8
)

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id,
)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
from accelerate import Accelerator

accelerator = Accelerator()
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

In [ ]:
from transformers import get_scheduler

num_train_epochs = 3
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

In [ ]:
api.create_repo(
    repo_id="gugukaka/bert-finetuned-ner-accelerate",
    repo_type="model",
    exist_ok=True
)

#training loop

In [ ]:
def postprocess(predictions, labels):
    predictions = predictions.detach().cpu().clone().numpy()
    labels = labels.detach().cpu().clone().numpy()

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_name[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_name[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    return true_labels, true_predictions

In [ ]:
from tqdm.auto import tqdm
import torch

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_train_epochs):
    # Training
    model.train()
    for batch in train_dataloader:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    # Evaluation
    model.eval()
    for batch in eval_dataloader:
        with torch.no_grad():
            outputs = model(**batch)

        predictions = outputs.logits.argmax(dim=-1)
        labels = batch["labels"]

        # Necessary to pad predictions and labels for being gathered
        predictions = accelerator.pad_across_processes(predictions, dim=1, pad_index=-100)
        labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)

        predictions_gathered = accelerator.gather(predictions)
        labels_gathered = accelerator.gather(labels)

        true_predictions, true_labels = postprocess(predictions_gathered, labels_gathered)
        metric.add_batch(predictions=true_predictions, references=true_labels)

    results = metric.compute()
    print(
        f"epoch {epoch}:",
        {
            key: results[f"overall_{key}"]
            for key in ["precision", "recall", "f1", "accuracy"]
        },
    )

    # Save and upload
    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)

    unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)

    if accelerator.is_main_process:
        tokenizer.save_pretrained(output_dir)
        repo.push_to_hub(
            commit_message=f"Training in progress epoch {epoch}",
            blocking=False
        )